# Entrenamiento YOLOv8s-seg — Segmentación de Residuos (TACO)

**Visión por Computadora II — CEIA / UBA**

Este notebook reproduce el fine-tuning en dos etapas del modelo reportado en el paper.
Diseñado para correr en **Google Colab con GPU T4** (gratis).

### Pasos
1. Verificar GPU
2. Instalar dependencias
3. Montar Google Drive (para guardar los pesos al final)
4. Descargar dataset TACO desde Zenodo (~2.7 GB)
5. Preparar dataset en formato YOLO-seg
6. Entrenamiento — Etapa 1 (backbone congelado)
7. Entrenamiento — Etapa 2 (fine-tuning completo)
8. Evaluación y métricas
9. Guardar `best.pt` en Drive

> **Tip:** En `Entorno de ejecución → Cambiar tipo de entorno de ejecución` elegí **GPU T4** antes de empezar.

## 1. Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No se detectó GPU NVIDIA. Verificá que seleccionaste GPU en el entorno de ejecución.')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Instalar dependencias

In [ ]:
# ultralytics incluye PyTorch, torchvision y todas las dependencias necesarias
# tqdm y requests ya vienen en Colab
%pip install -q ultralytics==8.4.138

## 3. Montar Google Drive

Los pesos `best.pt` se guardan en `Mi unidad/residuos_yolo/` al final del entrenamiento.
Si no querés usar Drive podés saltear esta celda; los archivos quedan en `/content/` hasta que se cierre la sesión.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/residuos_yolo'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Los pesos se guardarán en: {DRIVE_OUTPUT}')

## 4. Descargar dataset TACO

Descarga `TACO.zip` (~2.7 GB) del mirror de Zenodo y extrae las imágenes.
También clona el repo `pedropro/TACO` para obtener `annotations.json`.

**Tiempo estimado en Colab:** ~5 minutos (depende de la velocidad del servidor).

In [ ]:
import hashlib
import zipfile
from pathlib import Path
import requests
from tqdm.notebook import tqdm

# Rutas base
BASE_DIR   = Path('/content')
TACO_DIR   = BASE_DIR / 'TACO'
IMAGES_DIR = TACO_DIR / 'data'
ZIP_PATH   = BASE_DIR / 'TACO_zenodo.zip'

ZENODO_URL  = 'https://zenodo.org/api/records/3587843/files/TACO.zip/content'
ZENODO_MD5  = 'e9149407d883e21a8d224feef8210920'
TACO_REPO   = 'https://github.com/pedropro/TACO.git'

# 1) Clonar repo de TACO (solo para obtener annotations.json)
if not TACO_DIR.exists():
    print('Clonando repositorio TACO (anotaciones)...')
    !git clone --depth 1 {TACO_REPO} {TACO_DIR}
else:
    print('✅ Repositorio TACO ya existe.')

# 2) Descargar TACO.zip de Zenodo
def md5_of(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

zip_ok = False
if ZIP_PATH.exists():
    print('Verificando zip existente...')
    zip_ok = md5_of(ZIP_PATH) == ZENODO_MD5
    print('✅ Zip completo.' if zip_ok else '⚠️  Zip incompleto, se vuelve a descargar.')

if not zip_ok:
    print('Descargando TACO.zip desde Zenodo (~2.7 GB)...')
    tmp = ZIP_PATH.with_suffix('.part')
    with requests.get(ZENODO_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        with open(tmp, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc='TACO.zip') as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    print('Verificando MD5...')
    assert md5_of(tmp) == ZENODO_MD5, '⚠️  MD5 no coincide. Volvé a correr la celda.'
    tmp.rename(ZIP_PATH)
    print('✅ Descarga completa y verificada.')

# 3) Extraer imágenes
print('Extrayendo imágenes...')
extraidas = 0
with zipfile.ZipFile(ZIP_PATH) as z:
    miembros = [
        m for m in z.infolist()
        if m.filename.startswith('TACO/data/batch_')
        and not m.is_dir()
        and not Path(m.filename).name.startswith('.')
    ]
    for m in tqdm(miembros, desc='Extrayendo'):
        destino = IMAGES_DIR / Path(m.filename).relative_to('TACO/data')
        if destino.exists() and destino.stat().st_size == m.file_size:
            continue
        destino.parent.mkdir(parents=True, exist_ok=True)
        with z.open(m) as src, open(destino, 'wb') as dst:
            while chunk := src.read(1 << 20):
                dst.write(chunk)
        extraidas += 1

n_total = sum(1 for p in IMAGES_DIR.glob('batch_*/*') if p.is_file())
print(f'✅ {extraidas} imágenes nuevas extraídas. Total en disco: {n_total}')

# Liberar espacio borrando el zip
ZIP_PATH.unlink()
print('   Zip eliminado para liberar espacio.')

## 5. Preparar dataset en formato YOLO-seg

Convierte las anotaciones COCO (60 categorías finas) a **6 clases macro**
y genera los archivos `.txt` de YOLO-seg + `data.yaml`.

Split estratificado 80/20 por clase más escasa → **1202 train / 298 val**.

In [ ]:
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

ANNOTATIONS_PATH = IMAGES_DIR / 'annotations.json'
OUTPUT_DIR = BASE_DIR / 'taco_yolo'
VAL_SPLIT  = 0.2
RANDOM_SEED = 42

# Clases macro (el orden define el class_id en YOLO)
MACRO_CLASSES = ['plastico', 'papel_carton', 'vidrio', 'metal', 'organico', 'otros']

# Mapeo supercategoría TACO -> clase macro
SUPERCATEGORY_TO_MACRO = {
    'Plastic bag & wrapper': 'plastico',
    'Plastic container':     'plastico',
    'Other plastic':         'plastico',
    'Straw':                 'plastico',
    'Styrofoam piece':       'plastico',
    'Lid':                   'plastico',
    'Bottle':                'plastico',
    'Bottle cap':            'plastico',
    'Cup':                   'plastico',
    'Paper':                 'papel_carton',
    'Paper bag':             'papel_carton',
    'Carton':                'papel_carton',
    'Glass jar':             'vidrio',
    'Broken glass':          'vidrio',
    'Can':                   'metal',
    'Aluminium foil':        'metal',
    'Metal bottle cap':      'metal',
    'Pop tab':               'metal',
    'Scrap metal':           'metal',
    'Food waste':            'organico',
    'Cigarette':             'otros',
    'Unlabeled litter':      'otros',
    'Blister pack':          'otros',
    'Squeezable tube':       'otros',
    'Battery':               'otros',
    'Shoe':                  'otros',
}

# --- Limpiar salida anterior si existe ---
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print('Directorio taco_yolo anterior eliminado.')

random.seed(RANDOM_SEED)
print('Cargando anotaciones COCO...')
with open(ANNOTATIONS_PATH) as f:
    coco = json.load(f)

# Mapeo category_id -> clase macro
cat_id_to_macro = {}
for cat in coco['categories']:
    supercat = cat.get('supercategory', '')
    cat_id_to_macro[cat['id']] = SUPERCATEGORY_TO_MACRO.get(supercat, 'otros')

macro_to_class_id = {name: i for i, name in enumerate(MACRO_CLASSES)}
images_by_id = {img['id']: img for img in coco['images']}
anns_by_image = defaultdict(list)
for ann in coco['annotations']:
    anns_by_image[ann['image_id']].append(ann)

# Solo imágenes descargadas
valid_ids = [
    iid for iid in anns_by_image
    if (IMAGES_DIR / images_by_id[iid]['file_name']).exists()
]

# Split estratificado por clase más escasa
total_per_macro = defaultdict(int)
image_macros = {}
for iid in valid_ids:
    macros = {cat_id_to_macro.get(a['category_id'], 'otros') for a in anns_by_image[iid]}
    image_macros[iid] = macros
    for m in macros:
        total_per_macro[m] += 1

def clase_mas_rara(iid):
    macros = image_macros[iid]
    return min(macros, key=lambda m: total_per_macro[m]) if macros else 'otros'

grupos = defaultdict(list)
for iid in valid_ids:
    grupos[clase_mas_rara(iid)].append(iid)

val_ids = set()
for macro, ids_grupo in grupos.items():
    ids_grupo = ids_grupo.copy()
    random.shuffle(ids_grupo)
    n_val = max(1, int(len(ids_grupo) * VAL_SPLIT)) if len(ids_grupo) > 1 else 0
    val_ids.update(ids_grupo[:n_val])

# Crear directorios
for split in ['train', 'val']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Convertir y copiar
counts_by_split = {'train': defaultdict(int), 'val': defaultdict(int)}
skipped = 0

for image_id, anns in tqdm(anns_by_image.items(), desc='Convirtiendo'):
    img_info = images_by_id[image_id]
    src_img = IMAGES_DIR / img_info['file_name']
    if not src_img.exists():
        continue
    split = 'val' if image_id in val_ids else 'train'
    dst_name = f"{image_id}_{src_img.name}"
    dst_img = OUTPUT_DIR / 'images' / split / dst_name
    if not dst_img.exists():
        shutil.copy2(src_img, dst_img)

    w, h = img_info['width'], img_info['height']
    lines = []
    for ann in anns:
        macro = cat_id_to_macro.get(ann['category_id'], 'otros')
        cid = macro_to_class_id[macro]
        for poly in (ann.get('segmentation') or []):
            if len(poly) < 6:
                skipped += 1
                continue
            coords = ' '.join(
                f"{poly[i]/w:.6f} {poly[i+1]/h:.6f}"
                for i in range(0, len(poly), 2)
            )
            lines.append(f'{cid} {coords}')
            counts_by_split[split][macro] += 1

    label_path = OUTPUT_DIR / 'labels' / split / f"{image_id}_{src_img.stem}.txt"
    label_path.write_text('\n'.join(lines))

# Eliminar cache de Ultralytics si existiera
for cache in OUTPUT_DIR.rglob('*.cache'):
    cache.unlink()

# data.yaml
DATA_YAML = OUTPUT_DIR / 'data.yaml'
DATA_YAML.write_text(
    f"path: {OUTPUT_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    f"nc: {len(MACRO_CLASSES)}\n"
    f"names: {MACRO_CLASSES}\n"
)

n_train = len(valid_ids) - len(val_ids & set(valid_ids))
n_val   = len(val_ids & set(valid_ids))
print(f'\n✅ Dataset listo: {n_train} train / {n_val} val')
print('\nInstancias por clase (train / val):')
for cls in MACRO_CLASSES:
    t = counts_by_split['train'].get(cls, 0)
    v = counts_by_split['val'].get(cls, 0)
    print(f'  {cls:15s}: train={t:5d}  val={v:4d}')

## 6. Entrenamiento — Etapa 1 (backbone congelado)

Entrena solo las capas de detección/segmentación con el backbone congelado (`freeze=10`).
Esto permite una adaptación rápida al dominio de residuos sin romper las features pre-entrenadas.

**Config igual al modelo reportado en el paper:**
- `imgsz=960`, `batch=8`, `lr0=1e-3`, `freeze=10`, `epochs=20`

In [ ]:
from ultralytics import YOLO

# Hiperparámetros — igual al notebook 00 que produjo el modelo reportado
IMGSZ   = 960
BATCH_1 = 8
EPOCHS_1 = 20
PROJECT  = '/content/runs'
NAME_1   = 'residuos_etapa1'

model = YOLO('yolov8s-seg.pt')  # descarga automática del checkpoint base de ImageNet

print('=== Etapa 1: backbone congelado ===')
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_1,
    imgsz=IMGSZ,
    batch=BATCH_1,
    device='0',          # GPU CUDA (índice 0)
    project=PROJECT,
    name=NAME_1,
    freeze=10,
    lr0=1e-3,
    exist_ok=True,
    verbose=True,
)

## 7. Entrenamiento — Etapa 2 (fine-tuning completo)

Retoma el mejor checkpoint de la Etapa 1 y desbloquea parcialmente el backbone (`freeze=5`).
LR más bajo para no destruir las features aprendidas.

**Config igual al modelo reportado en el paper:**
- `imgsz=960`, `batch=4`, `lr0=3e-4`, `freeze=5`, `epochs=20`
- `copy_paste=0.5`, `mixup=0.2` (augmentación adicional)

In [ ]:
from pathlib import Path

BATCH_2  = 4
EPOCHS_2 = 20
NAME_2   = 'residuos_etapa2'

best_stage1 = Path(PROJECT) / NAME_1 / 'weights' / 'best.pt'
assert best_stage1.exists(), f'No se encontró {best_stage1}. ¿Corrió la Etapa 1?'

model2 = YOLO(str(best_stage1))

print('=== Etapa 2: fine-tuning parcialmente descongelado ===')
results = model2.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_2,
    imgsz=IMGSZ,
    batch=BATCH_2,
    device='0',
    project=PROJECT,
    name=NAME_2,
    freeze=5,
    lr0=3e-4,
    copy_paste=0.5,
    mixup=0.2,
    exist_ok=True,
    verbose=True,
)

best_final = Path(PROJECT) / NAME_2 / 'weights' / 'best.pt'
print(f'\n✅ Entrenamiento finalizado. Mejor checkpoint: {best_final}')

## 8. Evaluación sobre el conjunto de validación

In [ ]:
print('Evaluando sobre val...')
metrics = model2.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    device='0',
)

print('\n=== Métricas de validación ===')
print(f'  box mAP50:    {metrics.box.map50:.3f}')
print(f'  box mAP50-95: {metrics.box.map:.3f}')
print(f'  mask mAP50:   {metrics.seg.map50:.3f}')
print(f'  mask mAP50-95:{metrics.seg.map:.3f}')
print()
print('mAP50 mask por clase:')
MACRO_CLASSES = ['plastico', 'papel_carton', 'vidrio', 'metal', 'organico', 'otros']
for cls, v in zip(MACRO_CLASSES, metrics.seg.maps):
    print(f'  {cls:15s}: {v:.3f}')

## 9. Guardar `best.pt` en Google Drive

Copia el checkpoint final a `Mi unidad/residuos_yolo/best.pt` para que persista
después de que cierre la sesión de Colab.

Si salteaste el montaje de Drive en el paso 3, descargá el archivo manualmente
desde el panel de archivos de Colab (ícono de carpeta a la izquierda).

In [ ]:
import shutil
from pathlib import Path

best_final = Path(PROJECT) / NAME_2 / 'weights' / 'best.pt'

# --- Guardar en Drive ---
drive_dest = Path(DRIVE_OUTPUT) / 'best.pt'
shutil.copy2(best_final, drive_dest)
print(f'✅ Pesos guardados en Drive: {drive_dest}')

# --- También copiar los resultados del entrenamiento ---
results_dest = Path(DRIVE_OUTPUT) / 'resultados_etapa2'
shutil.copytree(Path(PROJECT) / NAME_2, results_dest, dirs_exist_ok=True)
print(f'✅ Resultados completos (curvas, matrices de confusión) en: {results_dest}')

In [ ]:
# Alternativa: descargar best.pt directamente al navegador (sin Drive)
from google.colab import files
files.download(str(best_final))

---
## Notas

**Resultados de referencia (modelo reportado en el paper)** — entrenado con esta misma config en Apple M4 Pro (MPS):

| Métrica         | Valor |
|-----------------|-------|
| box mAP50       | 0.251 |
| box mAP50-95    | 0.193 |
| mask mAP50      | 0.223 |
| mask mAP50-95   | 0.156 |

mAP50 mask por clase: `plastico` 0.459 · `papel_carton` 0.390 · `metal` 0.364 · `otros` 0.115 · `vidrio` ~0 · `organico` ~0

**Clases con bajo desempeño:** `vidrio` y `organico` tienen muy pocas instancias en TACO (6 y 8 en val respectivamente), lo que hace que su mAP sea prácticamente cero.

**Usar el modelo entrenado en la app:** copiá `best.pt` a la carpeta del repo y corré:
```bash
uv run streamlit run app/app.py
```
La app lo busca en `runs/segment/residuos_yolov8seg_etapa2/weights/best.pt` por defecto,
o podés especificar la ruta en el sidebar.